# Regression with PyTorch V1.3 (oracle) — Income, deeper MLP, k-fold validation (k=5)

**Version:** V1.3 (k-fold validation — deeper MLP). **Role:** reference oracle for the V1.3 scratch
twin. Identical `RegDeepMLP` architecture and hypers as `regression_pytorch_V1_1.ipynb`
(`FC(13→64)→BN→ReLU→Dropout(0.2)→FC(64→64)→BN→(+residual)→ReLU→Dropout(0.2)→FC(64→1)`, z-space MSE,
Adam+cosine+wd(1e-4), 100 epochs), copied verbatim and wrapped in a 5-fold plain `KFold` loop over the
full 2,212-row dropped-target set instead of one fixed 80/20 split — this is the single-split-vs-small-n
robustness check for V1.1's deeper MLP, pulled out into its own standalone version so it sits as a
sibling of V2.1/V3.1 rather than appended inside V1.1's own notebook. Weights → `./weights_V1_3/`.

In [1]:
import json, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs("weights_V1_3", exist_ok=True)
print("torch:", torch.__version__)

torch: 2.14.1+cu130


## 1. LOAD DATA (identical to V1.1)

In [2]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
df = df.dropna(subset=["Income"]).copy()
print("cleaned:", df.shape, "| Income median:", round(df["Income"].median()))

REG_FEATURES = [
    "NumCatalogPurchases", "MntMeatProducts", "MntWines", "NumWebVisitsMonth",
    "MntSweetProducts", "MntFruits", "NumWebPurchases", "MntGoldProds",
    "NumStorePurchases", "MntFishProducts", "Kidhome", "Age", "NumDealsPurchases",
]
X_full = df[REG_FEATURES].values
y_full = df["Income"].astype(float).values
print(f"features: {len(REG_FEATURES)} dims (all numeric)")

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.float32)).view(-1, 1)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.y[i]

cleaned: (2212, 28) | Income median: 51371
features: 13 dims (all numeric)


## 2. MODEL: RegDeepMLP (identical to V1.1, He init)

In [3]:
class RegDeepMLP(nn.Module):
    def __init__(self, in_dim=13, h=64, p=0.2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, h)
        self.bn1 = nn.BatchNorm1d(h)
        self.fc2 = nn.Linear(h, h)
        self.bn2 = nn.BatchNorm1d(h)
        self.fc3 = nn.Linear(h, 1)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(p)
        for fc, nl in ((self.fc1, "relu"), (self.fc2, "relu"), (self.fc3, "linear")):
            nn.init.kaiming_normal_(fc.weight, nonlinearity=nl)
            nn.init.zeros_(fc.bias)
    def forward(self, x):
        d1 = self.drop(self.relu(self.bn1(self.fc1(x))))
        s = self.bn2(self.fc2(d1)) + d1
        d2 = self.drop(self.relu(s))
        return self.fc3(d2)

device = torch.device("cpu")

## K-fold training (k=5) — fresh RegDeepMLP per fold, no checkpointing

Same convention as the V1.1 single split: each fold trains for a fixed 100 epochs (no best-epoch
checkpoint selection) and is evaluated on its own held-out fold at the end. Each fold re-fits its own
feature scaler and target scaler on that fold's training rows only, and trains a **fresh** model for
the same 100 epochs. Saved to `weights_V1_3/regression_pytorch_kfold_metrics.json`.

In [4]:
K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=SEED)
fold_rows = []
for fold, (tr_idx, va_idx) in enumerate(kf.split(X_full), start=1):
    Xtr_f, Xva_f = X_full[tr_idx], X_full[va_idx]
    ytr_f, yva_f = y_full[tr_idx], y_full[va_idx]
    sc_f = StandardScaler().fit(Xtr_f)
    Xtr_fs, Xva_fs = sc_f.transform(Xtr_f), sc_f.transform(Xva_f)
    ysc_f = StandardScaler().fit(ytr_f.reshape(-1, 1))
    ytr_fz = ysc_f.transform(ytr_f.reshape(-1, 1)).ravel()
    train_ds_f = TabularDataset(Xtr_fs, ytr_fz)
    train_loader_f = DataLoader(train_ds_f, batch_size=64, shuffle=True,
                                 generator=torch.Generator().manual_seed(SEED + fold))
    torch.manual_seed(SEED + fold)
    m = RegDeepMLP(in_dim=len(REG_FEATURES)).to(device)
    crit_f = nn.MSELoss()
    opt_f = torch.optim.Adam(m.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4)
    sch_f = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f, T_max=100)
    for epoch in range(100):
        m.train()
        for xb, yb in train_loader_f:
            xb, yb = xb.to(device), yb.to(device)
            opt_f.zero_grad()
            crit_f(m(xb), yb).backward()
            opt_f.step()
        sch_f.step()
    m.eval()
    with torch.no_grad():
        Xv = torch.from_numpy(Xva_fs.astype(np.float32)).to(device)
        pred_z = m(Xv).cpu().numpy().ravel()
    pred_raw = ysc_f.inverse_transform(pred_z.reshape(-1, 1)).ravel()
    mse_f = float(mean_squared_error(yva_f, pred_raw))
    fold_rows.append({
        "fold": fold,
        "test_loss_MSE_raw": round(mse_f, 1),
        "rmse": round(float(np.sqrt(mse_f)), 1),
        "mae": round(float(mean_absolute_error(yva_f, pred_raw)), 1),
        "r2": round(float(r2_score(yva_f, pred_raw)), 4),
    })
    print(f"fold {fold}: RMSE={fold_rows[-1]['rmse']:.1f} R2={fold_rows[-1]['r2']:.4f}")

kfold_df = pd.DataFrame(fold_rows)
summary = {k: {"mean": round(float(kfold_df[k].mean()), 4), "std": round(float(kfold_df[k].std()), 4)}
           for k in ("test_loss_MSE_raw", "rmse", "mae", "r2")}
print(kfold_df.to_string(index=False))
print(json.dumps(summary, indent=2))
with open("weights_V1_3/regression_pytorch_kfold_metrics.json", "w") as f:
    json.dump({"k": K, "folds": fold_rows, "summary": summary}, f, indent=2)

fold 1: RMSE=10464.4 R2=0.7695


fold 2: RMSE=11702.6 R2=0.6935


fold 3: RMSE=9268.1 R2=0.8084


fold 4: RMSE=11680.4 R2=0.7231


fold 5: RMSE=9898.9 R2=0.7818
 fold  test_loss_MSE_raw    rmse    mae     r2
    1        109503848.2 10464.4 6616.2 0.7695
    2        136951787.7 11702.6 7056.6 0.6935
    3         85898380.8  9268.1 6622.8 0.8084
    4        136431837.6 11680.4 7103.4 0.7231
    5         97987957.9  9898.9 6942.3 0.7818
{
  "test_loss_MSE_raw": {
    "mean": 113354762.44,
    "std": 22881171.9219
  },
  "rmse": {
    "mean": 10602.88,
    "std": 1080.143
  },
  "mae": {
    "mean": 6868.26,
    "std": 234.5365
  },
  "r2": {
    "mean": 0.7553,
    "std": 0.0463
  }
}
